# ⚡ Crushing the Engine: Barrier Options & Celery HPC
A Down-and-Out Barrier Option contains a step-function discontinuity. Calculating this accurately for institutional pricing requires massive trajectories. 

If a payload exceeds **50,000,000 total computational steps**, Prometheus will automatically intercept it to prevent HTTP timeouts and route it to our isolated C++ Celery Cluster.

Let's price a Barrier Option with `N = 1,000,000` paths and `M = 252` steps. 
* Total Steps: **252,000,000** (252 Million floating-point path evolutions).
* Cost: **1.008 Credits**.
* Execution: Asynchronous Polling.

In [ ]:
import requests
import uuid
import time

API_KEY = "pmt_live_your_secure_api_key_here"
BASE_URL = "https://api.prometheusquantengine.com/api/v1/simulations"
TASK_URL = "https://api.prometheusquantengine.com/api/v1/simulations/task"

headers = {
    "X-API-Key": API_KEY,
    "Idempotency-Key": str(uuid.uuid4()),
    "Content-Type": "application/json"
}

payload = {
    "simulation_type": "Barrier",
    "s_0": 100.0,
    "strike": 100.0,
    "volatility": 0.25,
    "time_to_maturity": 1.0,
    "risk_free_rate": 0.05,
    "option_type": "Put",
    "n_simulations": 1000000, # 1 Million Paths
    "m_steps": 252,           # Daily observations
    "barrier_type": "DownAndOut",
    "barrier_level": 85.0
}

print("Dispatching 252,000,000 computational steps to Prometheus HPC...")
response = requests.post(BASE_URL, json=payload, headers=headers)
data = response.json()

# Notice it returns a 201 Created with a task_id, NOT a fair_value.
task_id = data.get("task_id")
print(f"Engine Response: {data.get('message')}")
print(f"Task Ticket ID: {task_id}\n")

# Long Polling Protocol
print("Initiating Asynchronous Polling...")
while True:
    task_resp = requests.get(f"{TASK_URL}/{task_id}", headers={"X-API-Key": API_KEY})
    task_data = task_resp.json()
    status = task_data.get("status")
    
    if status == "SUCCESS":
        print("\n✅ COMPUTATION COMPLETE")
        print(f"Fair Value: {task_data.get('fair_value')}")
        print(f"Simulation ID: {task_data.get('simulation_id')}")
        break
    elif status in ["FAILURE", "REVOKED"]:
        print("\n❌ ENGINE FAILURE. Credits automatically refunded to escrow.")
        break
    
    print(f"[{time.strftime('%H:%M:%S')}] Cluster Status: {status}. Awaiting C++ threads...")
    time.sleep(2)

### 🏁 Conclusion
Attempting to process 252 Million stochastic steps inside a native Python/NumPy kernel would lock your machine, bloat your RAM, and take minutes to resolve. 

By leveraging Prometheus, you offloaded the entire matrix to AWS x86 optimized CPUs in the cloud, retrieving institutional-grade pricing in seconds. 
* **Go to `prometheusquantengine.com/dashboard` to check your updated ledger and simulation history.**